In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
vipoooool_new_plant_diseases_dataset_path = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')

print('Data source import complete.')


In [ ]:
#IMPORTING LIBRARIES
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# **MOTIVATION**

![LEAF](http://www.mobindustry.net/wp-content/uploads/1_IbJF_6mRTMsG9gL0j8uz5Q.jpeg)


*  **As farmers and agriculture field are the important part of our life , farmers are the root level building blocks in the economy of any country . They work really heard for a whole season to grow a specific crop for survival of his family**

* **Sometimes these crops on which he dedicated his whole 3-6 months to nurture these crops got disease as result of which they can't sell their crops on the price he was expecting**

*  **And He thinks if he knew these if he knew the plant disease before hand , he can use spefic pesticides and fertilizers to get over these disease**

*  **What if we can use deep learning techniques to help famers to know about specific disease , so that they can be ready before harvestifying their crops**

*  **Well I have implemented Resnet50 to detect the disease , but If we want model to be deploy inside a mobile app then mobilenet architecture will be suitable**

# DATASET

1. **We have 38 classes of plant disease images which contains 70295 images in training set and 17572 in valid set**

2. **Each class contains average of 1700-1800 number of images to work upon**

3. **Each image is of size= (256,256,3)**


In [ ]:
path='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
plt.figure(figsize=(70,70))
count=0
plant_names=[]
total_images=0
for i in os.listdir(path):
  count+=1
  plant_names.append(i)
  plt.subplot(7,7,count)

  images_path=os.listdir(path+"/"+i)
  print("Number of images of "+i+":",len(images_path),"||",end=" ")
  total_images+=len(images_path)

  image_show=plt.imread(path+"/"+i+"/"+images_path[0])

  plt.imshow(image_show)
  plt.xlabel(i)

  plt.xticks([])
  plt.yticks([])


print("Total number of images we have",total_images)





In [ ]:
print(plant_names)
print(len(plant_names))

**IMPORTING NECESSARY LIBRARIES FOR TRAINING OF MODEL**

In [ ]:
import tensorflow
from tensorflow import keras
from keras.models import Sequential,load_model,Model
from keras.layers import Conv2D,MaxPool2D,AveragePooling2D,Dense,Flatten,ZeroPadding2D,BatchNormalization,Activation,Add,Input,Dropout,GlobalAveragePooling2D
from keras.optimizers import SGD
from keras.initializers import glorot_uniform
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint,EarlyStopping,ReduceLROnPlateau


# RESNET50 IMPLEMENTATION USING KERAS API

**WHAT IS RENSET MODEL?**

1. Over the last few years deep learning and machine learning are the most hottest topic in the tech industry because it has wide use in real life problems one of them is computer vision and prediction and many more and in this journey researchers( we should be very grateful to them) have created many deep learning architectures which can give us tremendous results in predicting the images

2. Some of architectures which are widely known and used are Alexnet,VGG16,VGG19,Resnet and many more and due to observations and results researchers thought more we increase number of layers (I am talking about deep learning layers such as Conv2D,MaxPool2D,GlobalAveragePooling2D etc.) the model will learn more complex features from images , but but but;) they were absolutely wrong

3. They got know that a 56 layer network is performing very bad than 20 layer network even on the training data , AS you can see from below image


![Graph](http://d1m75rqqgidzqn.cloudfront.net/wp-data/2020/09/09193619/11-696x235.png)

4. As we can see 56 layer network has greater training error than 20 layer network

5. What is the problem? Can we think of that , OK let me tell you this problem is called vanishing gradient problem in which weights are not updated while backward propagation , Oh you don't know abotu this let me help you with that . Visit here : https://youtu.be/JIWXbzRXk1I (Krish Naik) , make sure you will go understand it properly

**WHY RESNET?**

1. Here comes the question then why resnet? Yeah we have the answer to combat the problem of vanishing gradient problem , it has something called "skip connections" which solves the problem of vanishing gradient

2. Before moving forward let me tell you about our over achiever boy resnet ;)
     1. Won 1st place in the ILSVRC 2015 classification competition with a top-5 error rate of 3.57%
     2. Won the 1st place in ILSVRC and COCO 2015 competition in ImageNet Detection,                          ImageNetlocalization, Coco detection and Coco segmentation.
     3. Replacing VGG-16 layers in Faster R-CNN with ResNet-101. They observed relative improvements           of 28%
     4. Efficiently trained networks with 100 layers and 1000 layers also.
     


**About resnet**

1. Resnet architecture have 2 important blocks , first is identity block and second is convolution block

2. Let me first explain identity block: The identity block is the standard block used in ResNets and  corresponds to the case where the input activation has the same dimension as the output activation. You can see in below image

![Identity_block](http://machinelearningknowledge.ai/wp-content/uploads/2020/12/ResNet-Residual-Network-Keras-Implementation-Identity-Block.png)

3. Convolution Block : We can use this type of block when the input and output dimensions don’t match up. The difference with the identity block is that there is a CONV2D layer in the shortcut path.

![Convolution_block](http://machinelearningknowledge.ai/wp-content/uploads/2020/12/ResNet-Keras-Implementation-Convolutional-Block.png)

4. For better understanding you can visit this LINK: https://machinelearningknowledge.ai/keras-implementation-of-resnet-50-architecture-from-scratch/

5. Also visit here(this blog is legendary) : https://towardsdatascience.com/understanding-and-visualizing-resnets-442284831be8#:~:text=ResNet%20Layers,layers%20remains%20the%20same%20%E2%80%94%204.

1. **I Have used transfer learning here and also implemented Resnet50 from scratch below**

2. **I hope you know below steps  , if you don't let me know I will edit this notebook again**

In [ ]:

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
base_model_tf=ResNet50(include_top=False,weights='imagenet',input_shape=(224,224,3),classes=38)


In [ ]:
#Model building
base_model_tf.trainable=False

pt=Input(shape=(224,224,3))
func=tensorflow.cast(pt,tensorflow.float32)
x=preprocess_input(func) #This function used to zero-center each color channel wrt Imagenet dataset
model_resnet=base_model_tf(x,training=False)
model_resnet=GlobalAveragePooling2D()(model_resnet)
model_resnet=Dense(128,activation='relu')(model_resnet)
model_resnet=Dense(64,activation='relu')(model_resnet)
model_resnet=Dense(38,activation='softmax')(model_resnet)


model_main=Model(inputs=pt,outputs=model_resnet)
model_main.summary()

In [ ]:
#Image augmentation
train_datagen= ImageDataGenerator(shear_range=0.2,zoom_range=0.2,horizontal_flip=False,vertical_flip=False
                                  ,fill_mode='nearest',width_shift_range=0.2,height_shift_range=0.2)

val_datagen=ImageDataGenerator()

path_train='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'

path_valid='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid'

train= train_datagen.flow_from_directory(directory=path_train,batch_size=32,target_size=(224,224),
                                         color_mode='rgb',class_mode='categorical',seed=42)

valid=val_datagen.flow_from_directory(directory=path_valid,batch_size=32,target_size=(224,224),color_mode='rgb',class_mode='categorical')



In [ ]:
#CallBacks
es=EarlyStopping(monitor='val_accuracy',verbose=1,patience=7,mode='auto')
mc=ModelCheckpoint(filepath='/content',monitor='val_accuracy',verbose=1,save_best_only=True)
lr=ReduceLROnPlateau(monitor='val_accuracy',verbose=1,patience=5,min_lr=0.001)

In [ ]:
model_main.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
#Training
model_main.fit(train,validation_data=valid,epochs=30,steps_per_epoch=200,verbose=1,callbacks=[mc,es,lr])

In [ ]:
model_main.save("RESNET50_PLANT_DISEASE.h5")

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import cv2
from PIL import Image

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(model_main.history.history['loss'],color='b',label='Training loss')
plt.plot(model_main.history.history['val_loss'],color='r',label='Validation loss')
plt.xlabel("epochs")
plt.ylabel("loss_value")
plt.title("loss")


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(model_main.history.history['accuracy'],color='b',label='Training accuracy')
plt.plot(model_main.history.history['val_accuracy'],color='r',label='Validation accsuracy')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.title("accuracy graph")


**Well below is the implementation of Resnet50**

1. **Resnet50 have 5 stages in which each stage contains both convolution block as well as identity block**

![Resnet50](http://machinelearningknowledge.ai/wp-content/uploads/2020/12/ResNet-Keras-Implementation-Architecture.png)



2. **Also see from here(see 50 layer architecture) , I will suggest read blogs from above given links**

![Layers](http://test.neurohive.io/wp-content/uploads/2019/01/resnet-architectures-34-101.png)

1. **Do you know why I have not run below cells to get results, actually i have tried to run it but I was not getting even 50% accuracy on training set**

2. **You know why? Well I don't use preprocessing function , what is it ? This function is Preprocessed numpy.array or a tf.Tensor with type float32.**

3. **The images are converted from RGB to BGR, then each color channel is zero-centered with respect to the ImageNet dataset, without scaling**

4. **sometimes data is not zero-centered according to the imagenet dataset and we don't get good results , so tensoflow has provided us this preprocess input , you can see I have used it in above model**

5. **Read it from here:https://www.tensorflow.org/api_docs/python/tf/keras/applications/resnet/preprocess_input**

**RESNET50 CODE IMPLEMENTATION**

**I will comment it out below code and you can take reference from here and can implement your own resnet architecture . Trust it will be fun :)**

In [ ]:
'''def indentity_block(X,f,stage,filters,block):

    conv_base_name= 'res'+str(stage)+block+"_branch"
    bn_name="bn"+str(stage)+block+"_branch"

    F1,F2,F3=filters
    X_shortcut=X

    X=Conv2D(filters=F1, kernel_size=(1,1), padding='valid', strides=(1,1), name=conv_base_name+"2a",kernel_initializer=glorot_uniform(seed=0))(X)
    X=BatchNormalization(axis=3,name=bn_name+"2a")(X)
    X=Activation('relu')(X)

    X=Conv2D(filters=F2, kernel_size=(f,f), padding='same', strides=(1,1), name=conv_base_name+"2b",kernel_initializer=glorot_uniform(seed=0))(X)
    X=BatchNormalization(axis=3,name=bn_name+"2b")(X)
    X=Activation('relu')(X)

    X=Conv2D(filters=F3, kernel_size=(1,1), padding='valid', strides=(1,1), name=conv_base_name+"2c",kernel_initializer=glorot_uniform(seed=0))(X)
    X=BatchNormalization(axis=3,name=bn_name+"2c")(X)
    X=Add()([X,X_shortcut])

    X=Activation('relu')(X)

    return(X)'''


In [ ]:
'''def convolution_block(X,f,stage,filters,block,s=2):
    conv_base_name="res"+str(stage)+block+"_branch"
    bn_name="bn"+str(stage)+block+"_branch"

    F1,F2,F3=filters
    X_shortcut=X

    X=Conv2D(filters=F1, kernel_size=(1,1), padding='valid', strides=(s,s), name=conv_base_name+"2a",kernel_initializer=glorot_uniform(seed=0))(X)
    X=BatchNormalization(axis=3,name=bn_name+"2a")(X)
    X=Activation('relu')(X)

    X=Conv2D(filters=F2, kernel_size=(f,f), padding='same', strides=(1,1), name=conv_base_name+"2b",kernel_initializer=glorot_uniform(seed=0))(X)
    X=BatchNormalization(axis=3,name=bn_name+"2b")(X)
    X=Activation('relu')(X)

    X=Conv2D(filters=F3, kernel_size=(1,1), padding='valid', strides=(1,1), name=conv_base_name+"2c",kernel_initializer=glorot_uniform(seed=0))(X)
    X=BatchNormalization(axis=3,name=bn_name+"2c")(X)

    X_shortcut=Conv2D(filters=F3,kernel_size=(1,1),padding='valid',strides=(s,s),name=conv_base_name+"1",kernel_initializer=glorot_uniform(seed=0))(X_shortcut)
    X_shortcut=BatchNormalization(axis=3,name=bn_name+"1")(X_shortcut)

    X=Add()([X,X_shortcut])

    X=Activation('relu')(X)

    return(X)'''



In [ ]:
'''def resnet50(input_size=(224,224,3)):

    X_input=Input(input_size)

    X=ZeroPadding2D((3,3))(X_input)

    #STAGE 1
    X=Conv2D(filters=64,kernel_size=(7,7),strides=(2,2),kernel_initializer=glorot_uniform(seed=0),name='conv1')(X)
    X=BatchNormalization(axis=3,name='bn1')(X)
    X=Activation('relu')(X)
    X=MaxPool2D((3,3),strides=(2,2))(X)

    #STAGE 2
    X=convolution_block(X,f=3,filters=[64,64,256],block="a",s=1,stage=2)
    X=indentity_block(X,f=3,filters=[64,64,256],block='b',stage=2)
    X=indentity_block(X,f=3,filters=[64,64,256],block='c',stage=2)

    #STAGE 3
    X=convolution_block(X,f=3,filters=[128,128,512],block="a",s=2,stage=3)
    X=indentity_block(X,f=3,filters=[128,128,512],block='b',stage=3)
    X=indentity_block(X,f=3,filters=[128,128,512],block='c',stage=3)
    X=indentity_block(X,f=3,filters=[128,128,512],block='d',stage=3)


    #STAGE 4
    X=convolution_block(X,f=3,filters=[256,256,1024],block="a",s=2,stage=4)
    X=indentity_block(X,f=3,filters=[256,256,1024],block='b',stage=4)
    X=indentity_block(X,f=3,filters=[256,256,1024],block='c',stage=4)
    X=indentity_block(X,f=3,filters=[256,256,1024],block='d',stage=4)
    X=indentity_block(X,f=3,filters=[256,256,1024],block='e',stage=4)
    X=indentity_block(X,f=3,filters=[256,256,1024],block='f',stage=4)


    #STAGE 5
    X=convolution_block(X,f=3,filters=[512,512,2048],block="a",s=2,stage=5)
    X=indentity_block(X,f=3,filters=[512,512,2048],block='b',stage=5)
    X=indentity_block(X,f=3,filters=[512,512,2048],block='c',stage=5)

    X=AveragePooling2D(pool_size=(2,2),padding='same')(X)

    model=Model(inputs=X_input,outputs=X,name="RESNET50")

    return(model)'''



In [ ]:

'''base_model=resnet50(input_size=(224,224,3))
base_model.load_weights('/content/resnet50_weights_tf_dim_ordering_tf_kernels_notop(1).h5')'''

In [ ]:
'''model1=base_model.output
model1=Flatten()(model1)
model1=Dense(256,activation='relu',name='Dense1')(model1)
model1=Dropout(0.2)(model1)
model1=Dense(128,activation='relu',name='Dense1.1')(model1)
model1=Dropout(0.2)(model1)

model1=Dense(38,activation='softmax',name='Dense3')(model1)

main_model=Model(inputs=base_model.input,outputs=model1)
main_model.summary()'''

In [ ]:
'''base_model.trainable=False
for layer in main_model.layers:
  print(layer,layer.trainable)'''

# Last but not the least

1. **Give it a upvote if you love it , your upvote may be can help me to get a job :)**
2. **Will come up with more notebooks**
3. **Have any doubt please comment and ask , don't get appreciate If you really like my work**
4. **If you are beginner and won't able to understand the code , tell me I will edit this notebook or provide you resources to get better intuition**
5. **Thanks will meet on the next :)**